In [1]:
import numpy as np
import pandas as pd
import os

!pip install mlflow dagshub -q

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("DAGSHUB_TOKEN")
os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("DAGSHUB_USERNAME")

import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow")
print("MLflow connected!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 84.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"

test_transaction = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
test_identity = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

test = test_transaction.merge(test_identity, on="TransactionID", how="left")
del test_transaction, test_identity

test_ids = test["TransactionID"].copy()
print(f"Test shape: {test.shape}")
test.head()

Test shape: (506691, 433)


,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663549,18403224,31.95,W,10409,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3663550,18403263,49.00,W,4272,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3663551,18403310,171.00,W,4476,574.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3663552,18403310,284.95,W,10989,360.0,150.0,visa,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3663553,18403317,67.95,W,18018,452.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
loaded_pipeline = mlflow.sklearn.load_model("models:/ieee-fraud-best-model/latest")
print("Pipeline loaded from Model Registry")

predictions = loaded_pipeline.predict_proba(test)[:, 1]
print(f"Predictions: min={predictions.min():.4f}, max={predictions.max():.4f}, mean={predictions.mean():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud": predictions,
})
submission.to_csv("submission.csv", index=False)

print(f"\nsubmission.csv written: {submission.shape}")
submission.head()

Pipeline loaded from Model Registry


/tmp/ipykernel_57/3867785139.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_57/3867785139.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_57/3867785139.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_57/3867785139.py:64: PerformanceWarning: DataF

Predictions: min=0.0000, max=1.0000, mean=0.0213

submission.csv written: (506691, 2)


,TransactionID,isFraud
0,3663549,0.000029
1,3663550,0.000024
2,3663551,0.000070
3,3663552,0.000029
4,3663553,0.000013
